In [1]:
import qiskit
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

print("Qiskit Version:", qiskit.__version__)
simulator = AerSimulator()

def apply_constant_oracle(qc, n, value):
    if value == 1:
        qc.x(n)

def apply_parity_oracle(qc, n):
    for q in range(n):
        qc.cx(q, n)

def apply_truth_table_oracle(qc, n, outputs):
    for x, fx in enumerate(outputs):
        if fx == 1:
            bits = format(x, f"0{n}b")
            for q, bit in enumerate(bits):
                if bit == "0": qc.x(q)
            qc.mcx(list(range(n)), n)
            for q, bit in enumerate(bits):
                if bit == "0": qc.x(q)

def deutsch_jozsa(n, oracle_type="constant", constant_value=0, outputs=None, shots=1024):
    qc = QuantumCircuit(n+1, n)
    qc.x(n)
    qc.h(range(n+1))
    if oracle_type == "constant": apply_constant_oracle(qc,n,constant_value)
    elif oracle_type == "parity": apply_parity_oracle(qc,n)
    elif oracle_type == "truth_table": apply_truth_table_oracle(qc,n,outputs)
    else: raise ValueError("Unknown oracle type")
    qc.h(range(n))
    qc.measure(range(n),range(n))
    counts = simulator.run(qc,shots=shots).result().get_counts()
    return qc, counts

def classify_result(counts,n):
    return "Constant" if counts.get("0"*n,0)==sum(counts.values()) else "Balanced"

Qiskit Version: 2.5.1


In [2]:
for n in range(2,6):
    _,c=deutsch_jozsa(n,"constant",0,shots=1024)
    _,b=deutsch_jozsa(n,"parity",shots=1024)
    print(f"n={n}: constant -> {classify_result(c,n)}, balanced -> {classify_result(b,n)}")
    print("  Constant counts:",c)
    print("  Balanced counts:",b)

n=2: constant -> Constant, balanced -> Balanced
  Constant counts: {'00': 1024}
  Balanced counts: {'11': 1024}
n=3: constant -> Constant, balanced -> Balanced
  Constant counts: {'000': 1024}
  Balanced counts: {'111': 1024}
n=4: constant -> Constant, balanced -> Balanced
  Constant counts: {'0000': 1024}
  Balanced counts: {'1111': 1024}
n=5: constant -> Constant, balanced -> Balanced
  Constant counts: {'00000': 1024}
  Balanced counts: {'11111': 1024}
